In [ ]:
import pdfplumber
import fitz  # PyMuPDF
import os
import re
import docx
from docx import Document
#path to the pdf 
def extract_from_pdf_pdfplumber(file_path: str) -> str:
    """Extract text using pdfplumber (primary method)."""
    text_chunks = []

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=2)
            if text:
                text_chunks.append(text)

    return "\n".join(text_chunks)

def extract_from_pdf_pymupdf(file_path: str) -> str:
    """Fallback PDF extraction using PyMuPDF."""
    text_chunks = []

    with fitz.open(file_path) as doc:
        for page in doc:
            text = page.get_text("text")
            if text:
                text_chunks.append(text)

    return "\n".join(text_chunks)

def extract_from_docx(file_path: str) -> str:
    """Extract text from .docx file."""
    doc = Document(file_path)
    paragraphs = [para.text for para in doc.paragraphs if para.text.strip()]
    return "\n".join(paragraphs)



In [17]:
def clean_text(text: str) -> str:
    """
    Clean extracted resume text.
    Removes weird bullets, excessive whitespace, and hidden characters.
    """
    if not text:
        return ""

    # Normalize unicode quotes/dashes
    text = text.replace("\u2013", "-")
    text = text.replace("\u2014", "-")
    text = text.replace("\u2022", "-")  # bullet → dash
    text = text.replace("\xa0", " ")    # non-breaking space

    # Remove control characters
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", text)

    # Remove multiple spaces
    text = re.sub(r"[ ]{2,}", " ", text)

    # Remove excessive newlines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [18]:
def extract_text(file_path: str) -> str:
    """
    Main function:
    Detects file type and extracts raw text.
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    _, ext = os.path.splitext(file_path)
    ext = ext.lower()

    raw_text = ""

    if ext == ".pdf":
        try:
            raw_text = extract_from_pdf_pdfplumber(file_path)
            if not raw_text.strip():
                raise ValueError("pdfplumber returned empty text")
        except Exception:
            raw_text = extract_from_pdf_pymupdf(file_path)

    elif ext == ".docx":
        raw_text = extract_from_docx(file_path)

    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

    else:
        raise ValueError(f"Unsupported file format: {ext}")

    return clean_text(raw_text)

In [21]:
#file_path = "D:/cooding/ml/Mini-Project/CV_to_Portfolio_web/sample.pdf"
file_path = "D:/cooding/ml/Mini-Project/CV_to_Portfolio_web/aman_resume (2).docx"
text = extract_text(file_path)  
print(text)

NameError: name 'Document' is not defined